This is a notebook to visualize different properties of the dataset as we go about curating. Here, we use information from [InterPro](https://www.ebi.ac.uk/interpro/), an organization scheme for proteins that classifies them into families and predicts domains and important sites, using predictive models.

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl

from collections import defaultdict

from project.utils.strs import SEED, data_dir
from project.utils.functions import one_hot_polars_column, check_correlation_plotly
from project.utils.splitting import hierarchical_clustered_split


In [ ]:
subset_dir = data_dir / 'processed_subsets'
interpro_dir = subset_dir / 'interpro'
interpro_dir.mkdir(parents=True, exist_ok=True)

### Reading the annotated dataset

In [ ]:
# Read annotated dataset
annotated_data = 'uniprotkb_AND_model_organism_9606_2025_09_09_annotated.parquet.gz'
df = pl.read_parquet(data_dir / annotated_data)

In [ ]:
# Table to map interpro ids, types, and names
interpro_annotation_file = 'interpro_entry_list.txt'
interpro_names = pl.read_csv(data_dir / interpro_annotation_file, separator="\t")
interpro_names_dict = dict(zip(interpro_names['ENTRY_AC'], interpro_names['ENTRY_NAME']))
interpro_types_dict = dict(zip(interpro_names['ENTRY_AC'], interpro_names['ENTRY_TYPE']))

mmseqs_cols = [col for col in df.columns if ('mmseqs' in col)]
cols_to_keep = ['id', 'sequence'] + mmseqs_cols + ['split']

# Convert all terms higher than a given number of counts to one-hot, and drop rows that are all negative
onehot_df = one_hot_polars_column('InterPro', df, cols_to_keep[:-1], delimiter=';', occurrence_threshold=1, only_get_positive=True)

# Get which category each identifier belongs to
cat_cols = defaultdict(list)
for cat in interpro_names['ENTRY_TYPE'].unique().to_list():
    for c in onehot_df.columns[2:]:
        try:
            if interpro_types_dict[c] == cat:
                cat_cols[cat].append(c)
        except:
            continue

# Make dfs at different levels of organization
dfs = {}
for cat, ids in cat_cols.items():
    readable_ids = [interpro_names_dict[c] for c in ids]
    df_to_add = onehot_df.select(cols_to_keep[:-1] + ids)
    df_to_add.columns = [f"{c}__{interpro_names_dict[c].lower().replace(' ', '_')}" if (c in interpro_names_dict.keys()) else c for c in df_to_add.columns ]
    # Add to the storage dict after removing any rows that don't have any annotation for this category
    dfs[f'InterPro_{cat.lower().replace(' ', '_')}'] = df_to_add.filter(pl.sum_horizontal([col for col in df_to_add.columns if (df_to_add[col].dtype != pl.String)]) > 0)

Going to have to treat differrent levels of organization differently:
- Superfamily: enforce at least n members present in val, test set: we should be able to find members of the same superfamily that have different mmseqs representative members, and can put some into the test set, so we can afford to be a bit more stringent
- Family: enforce at least n members present in val, test set: we should be able to find members of the same family that have different mmseqs representative members, and can put some into the test set, so we can afford to be a bit more stringent
- Domain: enforce at least n members present in val, test set: domains are generally more specific, might have a hard time finding unrelated members with the same domain to put into the test set, so we can't afford to be too stringent
- Active site, binding site, ptm, repeat, conserved site: these are probably quite specific to a related group of proteins - we should probably adjust the clustering thresholds to be more permissive

In [ ]:
num_splits = 3

dfs_processed = {}
split_count_thresholds = {
    'InterPro_homologous_superfamily': 4, 
    'InterPro_family': 1,
    'InterPro_domain': 1, 
    'InterPro_active_site': 0, 
    'InterPro_binding_site': 0, 
    'InterPro_ptm': 0, 
    'InterPro_repeat': 0, 
    'InterPro_conserved_site': 0,
} # must have at least n samples in the val set and the test set

fine_grained_cats = [ 'InterPro_active_site', 'InterPro_binding_site', 'InterPro_ptm', 'InterPro_repeat', 'InterPro_conserved_site'] # Deals with proteins on a fine-grained level

for cat_name, onehot_df in dfs.items():
    for i in range(num_splits):
        if i !=0:
            appendix = '_split' + str(i)
        else:
            appendix = ''

        print(cat_name)
        split_count_threshold = split_count_thresholds[cat_name]

        if cat_name in fine_grained_cats:
            val_threshold_value = 50 # 40% identity minimum
            test_threshold_value = 30 #0.20
            val_threshold = f'mmseqs_0.{val_threshold_value}'
            test_threshold = f'mmseqs_0.{test_threshold_value}'
        else:
            val_threshold_value = 50 # 40% identity minimum
            test_threshold_value = 20 #0.20
            val_threshold = f'mmseqs_0.{val_threshold_value}'
            test_threshold = f'mmseqs_0.{test_threshold_value}'

        # Apply train-val-test splits
        onehot_df = pl.concat(hierarchical_clustered_split(onehot_df, 
                    val_threshold = val_threshold,
                    test_threshold = test_threshold,
                    val_ratio = 0.15,
                    test_ratio = 0.15,
                    seed = SEED+i))

        if cat_name not in fine_grained_cats:
            # Make sure that we have at least split_count_thresholds entities in the train / val set and remove the columns that don't
            test_df = onehot_df.filter(pl.col('split') == 'test')
            val_df = onehot_df.filter(pl.col('split') == 'val')
            target_cols = [c for c in onehot_df.columns if (test_df[c].dtype != pl.String)]
            standard_cols = [c for c in onehot_df.columns if (onehot_df[c].dtype == pl.String)]
            target_cols = [c for c in target_cols if ((test_df[c].sum() >= split_count_threshold) and (val_df[c].sum() >= split_count_threshold))]
            onehot_df = onehot_df.select(standard_cols + target_cols) # Select columns that 
            onehot_df = onehot_df.filter(pl.sum_horizontal([col for col in onehot_df.columns if (onehot_df[col].dtype != pl.String)]) > 0) # Drop any rows that now don't have annotations

        print('larger dataset ratio')
        print(onehot_df['split'].value_counts(normalize=True))

        #make a smaller version of the df
        # Make a dataset that's a subset for the checkpoints experiments - it has a max context length of 512 tokens.
        df_512 = onehot_df.filter(pl.col('sequence').str.len_chars() <= 512)
        print('after subsetting')
        print(df_512['split'].value_counts(normalize=True))
        df_512.write_parquet(interpro_dir / f"h_sapiens_proteome_interpro_{cat_name}_clustersplit_{test_threshold_value}_{val_threshold_value}_512_cutoff{appendix}.parquet.gz")

        # 4. Save the DataFrame
        dfs_processed[cat_name] = onehot_df

        if 'ptm' not in cat_name:
            onehot_df.write_parquet(interpro_dir / f"h_sapiens_proteome_interpro_{cat_name}_clustersplit_{test_threshold_value}_{val_threshold_value}{appendix}.parquet.gz")

Subsetting just to proteins of length 512 doesn't seem to meaningfully change the train:val:test composition

In [ ]:
for k, v in dfs_processed.items():
    print(f"{k}, proteins: {v.shape[0]}, annotation_columns: {len([c for c in v.columns if (v[c].dtype != pl.String)])}")

### Checking what terms are correlated with each other

In [ ]:
for cat in dfs_processed.keys():
    print(cat)
    try:
        fig = check_correlation_plotly(dfs_processed[cat], color_map='viridis', width = 1500, height=1500)
        fig.show()
    except:
        continue

There do not seem to be dangerous correlations (e.g. one category correlated with everything else); the correlations between different groups make sense, like immunoglobulin-like domain superfamily and immunoglobulin-like fold.